# GLM Analysis for Reconstructed fMRI

This notebook runs first-level GLM analysis, creates activation-map quality-control figures, and computes F2 scores across reconstruction methods.

Run the sections in order. Update paths and reconstruction specifications in the configuration cells before starting a new dataset.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, clear_output, display

NOTEBOOK_DIR = Path('/volatile/Caini/recon/glm_analysis')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from fmri_glm_toolkit import (
    GlobalConfig,
    ReconSpec,
    make_overlay_png,
    plot_bg_slices_png,
    run_batch,
    run_first_level_single_session,
)
from fmri_metric_toolkit import (
    plot_real_rivers_with_simulation_points,
    run_multi_base_pipeline,
)

## 2. Global configuration

The affine below corresponds to the 1219 acquisition. Replace it when processing a dataset with different geometry.

In [ ]:
OUTPUT_ROOT = Path('/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/nifti_1219')
DEFAULT_T1 = Path(
    '/volatile/Caini/000005_t1-mpr-moco-tra-iso1mm-moco_t1_mpr-moco_tra_iso1mm_20260224101132_5.nii'
)

cfg = GlobalConfig(
    father_out=str(OUTPUT_ROOT),
    default_t1_path=str(DEFAULT_T1),
    matlab_cmd=(
        '/volatile/Caini/stimulate/spm/spm_standalone/run_spm25.sh '
        '/usr/local/MATLAB_Runtime/R2024b/ script'
    ),
    tr=1.0,
    alpha_fpr_default=1e-3,
    cluster_th_default=30,
    cut_coords=(20, -56, 6),
    n_jobs=20,
    spm_jobtype='estimate',
    save_design_matrix=True,
)

AFFINE = np.array([
    [-1.46897663e-11, 3.0, 0.0, -96.0],
    [3.0, 1.46897663e-11, 0.0, -73.8785954],
    [0.0, 0.0, -3.0, 97.7907333],
    [0.0, 0.0, 0.0, 1.0],
])

## 3. Reconstruction inputs

Each `ReconSpec` identifies one reconstructed time series and its GLM parameters. Add or remove entries from `specs` as needed.

In [ ]:
def make_recon_spec(label, date_pattern, *, n_scans, tr, scale=1.0, overwrite=True):
    return ReconSpec(
        path={
            'base': '/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/recon_1219',
            'date': date_pattern,
            'suffix': label,
        },
        label=label,
        overwrite=overwrite,
        scale=scale,
        n_scans=n_scans,
        tr=tr,
        bg_mode='mean_func',
        bg_t1_path=cfg.default_t1_path,
        use_bg_complex=False,
        bg_policy='none',
    )

specs = [
    make_recon_spec(
        'R1_288_cg_orc', ['2026-07-05_01*'], n_scans=120, tr=2.0, scale=2.0
    ),
    make_recon_spec(
        'R1_298_cg_orc', ['2026-07-05_01*'], n_scans=120, tr=2.0, scale=2.0
    ),
]

pd.DataFrame([
    {
        'label': spec.label,
        'path': spec.path,
        'n_scans': spec.n_scans,
        'tr': spec.tr,
        'scale': spec.scale,
        'overwrite': spec.overwrite,
    }
    for spec in specs
])

## 4. Run first-level GLM

This step writes NIfTI files, motion-correction results, design matrices, z-maps, and QC products under `OUTPUT_ROOT`.

In [ ]:
results = run_batch(specs, cfg, AFFINE)

results_df = pd.DataFrame([
    {
        'label': result.label,
        'status': result.status,
        'n_frames': result.T,
        'threshold': result.threshold,
        'n_suprathreshold': result.n_supra,
        'max_z': result.max_z,
        'error': result.error,
    }
    for result in results
])
display(results_df)

## 5. Activation-map quality control

Generate one mosaic overlay for every completed GLM output. Existing reconstruction and z-map files are reused.

In [ ]:
spec_by_label = {spec.label: spec for spec in specs}
qc_rows = []

for out_dir in sorted(OUTPUT_ROOT.iterdir()):
    if not out_dir.is_dir():
        continue
    if not (out_dir / 'recon.nii').exists() or not (out_dir / 'z_map_glob.nii.gz').exists():
        continue

    spec = spec_by_label.get(
        out_dir.name,
        ReconSpec(path='', label=out_dir.name, bg_mode='mean_func'),
    )
    png = make_overlay_png(
        out_dir=out_dir,
        cfg=cfg,
        spec=spec,
        alpha=cfg.alpha_fpr_default,
        cluster_th=cfg.cluster_th_default,
        cut_coords=cfg.cut_coords,
        out_png=out_dir / 'QC_overlay_mean_func_alpha1e-3_cl30.png',
        layout='mosaic',
    )
    qc_rows.append({'label': out_dir.name, 'overlay': str(png)})

qc_df = pd.DataFrame(qc_rows)
display(qc_df)

### Optional interactive QC viewer

Run this cell in Jupyter to inspect different alpha and cluster-threshold settings without changing the GLM results.

In [ ]:
import ipywidgets as widgets

available_labels = sorted(
    path.name
    for path in OUTPUT_ROOT.iterdir()
    if path.is_dir() and (path / 'z_map_glob.nii.gz').exists()
)

if not available_labels:
    print('No completed z-maps were found under OUTPUT_ROOT.')
else:
    label_widget = widgets.Dropdown(
        options=available_labels,
        description='Recon',
        layout=widgets.Layout(width='700px'),
    )
    alpha_widget = widgets.FloatLogSlider(
        value=1e-3, base=10, min=-6, max=-1, step=0.1, description='Alpha'
    )
    cluster_widget = widgets.IntSlider(
        value=30, min=0, max=2000, step=5, description='Cluster'
    )
    qc_output = widgets.Output()

    def refresh_qc(*_):
        with qc_output:
            clear_output(wait=True)
            out_dir = OUTPUT_ROOT / label_widget.value
            spec = spec_by_label.get(
                label_widget.value,
                ReconSpec(path='', label=label_widget.value, bg_mode='mean_func'),
            )
            png = make_overlay_png(
                out_dir=out_dir,
                cfg=cfg,
                spec=spec,
                alpha=float(alpha_widget.value),
                cluster_th=int(cluster_widget.value),
                cut_coords=cfg.cut_coords,
            )
            print(f'Folder: {out_dir}')
            display(Image(filename=str(png)))

    for widget in (label_widget, alpha_widget, cluster_widget):
        widget.observe(refresh_qc, names='value')

    display(widgets.VBox([
        label_widget,
        widgets.HBox([alpha_widget, cluster_widget]),
        qc_output,
    ]))
    refresh_qc()

## 6. F2 analysis configuration

The two R1 anchors for each visit define the reference union mask. Verify that every anchor folder exists before running the analysis.

In [ ]:
F2_REFERENCE_T1 = Path(
    '/volatile/Caini/000005_t1-mpr-moco-tra-iso1mm-moco_t1_mpr-moco_tra_iso1mm_20260210102647_5.nii'
)
F2_BASES = [
    '/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/nifti_1219',
    '/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/nifti_0210',
    '/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/nifti_0224',
    '/volatile/Caini/mnt/topaze/scratch/results_3T_invivo/nifti_0306',
]
ANCHORS_BY_DATE = {
    '1219': ('R1_1_cg', 'R1_2_cg'),
    '0210': ('R1_56_cg', 'R1_66_cg'),
    '0224': ('R1_25_cg', 'R1_34_cg'),
    '0306': ('R1_265_cg', 'R1_271_cg'),
}
F2_THRESHOLD = {
    'alpha': 0.001,
    'height_control': 'fpr',
    'cluster_threshold': 30,
    'two_sided': True,
}

pd.DataFrame([
    {'date': date, 'anchor_a': anchors[0], 'anchor_b': anchors[1]}
    for date, anchors in ANCHORS_BY_DATE.items()
])

## 7. Compute F2 scores

This cell runs the metric pipeline and returns one record per reconstruction case.

In [ ]:
reference_img = nib.load(F2_REFERENCE_T1)

base_results, all_records = run_multi_base_pipeline(
    bases=F2_BASES,
    ref_img=reference_img,
    run_first_level_single_session=run_first_level_single_session,
    anchors_by_date=ANCHORS_BY_DATE,
    beta=2.0,
    thresh_kwargs=F2_THRESHOLD,
)

f2_df = pd.DataFrame(all_records)
required_columns = {'R', 'method', 'f2'}
missing_columns = required_columns.difference(f2_df.columns)
if missing_columns:
    raise KeyError(f'Missing F2 columns: {sorted(missing_columns)}')

f2_df['f2'] = pd.to_numeric(f2_df['f2'], errors='coerce')
f2_df = f2_df.dropna(subset=['R', 'method', 'f2'])
sort_columns = ['R', 'method'] + (['idx'] if 'idx' in f2_df.columns else [])
f2_df = f2_df.sort_values(sort_columns).reset_index(drop=True)
display(f2_df)

## 8. Export and plot F2 scores

In [ ]:
F2_OUTPUT_DIR = OUTPUT_ROOT / 'f2_analysis'
F2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

f2_csv = F2_OUTPUT_DIR / 'f2_scores.csv'
f2_png = F2_OUTPUT_DIR / 'f2_scores.png'
f2_df.to_csv(f2_csv, index=False)

f2_summary = (
    f2_df.groupby(['R', 'method'], as_index=False)['f2']
    .agg(mean='mean', std='std', sem='sem', n='count')
)

fig, ax = plt.subplots(figsize=(8, 5))
for method, group in f2_summary.groupby('method'):
    group = group.sort_values('R')
    ax.errorbar(
        group['R'],
        group['mean'],
        yerr=group['sem'].fillna(0),
        marker='o',
        capsize=4,
        linewidth=2,
        label=method,
    )

ax.set_xlabel('Acceleration factor R')
ax.set_ylabel('F2 score (beta=2)')
ax.set_title('GLM activation F2 score')
ax.set_xticks(sorted(f2_df['R'].unique()))
ax.grid(alpha=0.25)
ax.legend(title='Method')
fig.tight_layout()
fig.savefig(f2_png, dpi=300, bbox_inches='tight')
plt.show()

display(f2_summary)
print(f'Saved F2 records to {f2_csv}')
print(f'Saved F2 figure to {f2_png}')

## 9. Optional real-data and simulation comparison

In [ ]:
SIMULATION_F2_CSV = Path(
    '/volatile/Caini/mnt/topaze/scratch/simulate_3T_snake/results/outputs/f2_analysis/f2_details.csv'
)

plot_real_rivers_with_simulation_points(
    all_records,
    SIMULATION_F2_CSV,
    score_key='f2',
    title='Real-data and simulation F2 scores',
    error_mode='sem',
    river_point_color='red',
    include_sim_methods=('r1', 'cold', 'global'),
)
plt.show()